# The Green River Affair: Exploratory Data Analysis

**Group:** Epoch Fail

**Mission:** To analyze the provided datasets concerning the fish mortality events in the Green River and uncover the true cause, exonerating the wrongfully accused Myrtfolk.

This notebook is written as a reproducible scientific EDA report.

**What you will find here**
- How each dataset is loaded and cleaned (with explicit assumptions).
- Global visualizations (time series + distributions) to understand normal river behavior.
- Incident-by-incident windows (plots + numbers) to support evidence-based conclusions.
- A small scan for potential unreported events suggested by the data.

**Important:** Some datasets contain clearly invalid values (sensor failures / placeholders). We treat them as missing and keep a log of what we changed.

## 1. Introduction & Methodology

This report details the exploratory data analysis (EDA) performed on five datasets: fish mortality reports, water quality sensor data, meteorological data, river flow data, and nutrient sample data. 

Our approach is to (1) build a clean daily timeline, (2) understand typical seasonal behavior, (3) zoom in around fish mortality events, and (4) form evidence-based explanations. Data quality issues such as missing values, inconsistent formats, and sensor errors are explicitly documented and handled.

In [ ]:
# Imports & plotting defaults
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ModuleNotFoundError:
    # Allows running outside Jupyter (minimal fallback)
    def display(x):
        return x

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as e:
    raise ModuleNotFoundError("Missing dependency: matplotlib. Install with: pip install matplotlib seaborn") from e

try:
    import seaborn as sns
except ModuleNotFoundError as e:
    raise ModuleNotFoundError("Missing dependency: seaborn. Install with: pip install seaborn") from e

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

## Colab setup (Epoch Fail)
In Google Colab your working directory is usually `/content`. This section makes the notebook robust to where you placed the data.

Recommended workflow in Colab:
1. Get the repository into the runtime (e.g., clone it) so `data/` is available next to this notebook.
2. Run the cell below: it will auto-detect the data directory (and unzip if a legacy zip is present).

Alternative: mount Google Drive and point `DATA_DIR` to your Drive folder.

In [ ]:
# Colab/Local: locate data files and optionally unzip
import zipfile

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

required = [
    'fish_mortality_report.txt',
    'water_quality_sensors.csv',
    'meteo.json',
    'river_flow.xml',
    'nutrients.csv',
]

# If a zip is present (Colab upload), unzip once.
# In Colab, uploaded files land in /content and may be renamed (e.g. fish_mortality (1).zip).
zip_candidates = []

# Common expected names
for p in [Path('fish_mortality.zip'), Path('/content/fish_mortality.zip')]:
    if p.exists():
        zip_candidates.append(p)

# Any zip in /content / current directory (handles name changes)
if IN_COLAB:
    zip_candidates.extend(sorted(Path('/content').glob('*.zip')))
zip_candidates.extend(sorted(Path('.').glob('*.zip')))

# De-duplicate while keeping order
seen = set()
zip_candidates = [p for p in zip_candidates if not (str(p) in seen or seen.add(str(p)))]

def zip_has_required(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            names = set(zf.namelist())
        for f in required:
            if f in names:
                continue
            # support foldered zips: foo/fish_mortality_report.txt
            if not any(n.endswith('/' + f) for n in names):
                return False
        return True
    except Exception:
        return False

selected_zip = None
for p in zip_candidates:
    if zip_has_required(p):
        selected_zip = p
        break

# If no zip found but we're in Colab, offer an in-notebook upload
if selected_zip is None and IN_COLAB:
    print('No suitable zip found yet. Upload the dataset zip now (it may take a few seconds).')
    uploaded = files.upload()
    for name in uploaded.keys():
        p = Path(name)
        if p.suffix.lower() == '.zip' and zip_has_required(p):
            selected_zip = p
            break

UNPACK_DIR = Path('_unpacked')
if selected_zip is not None:
    marker = UNPACK_DIR / '._done'
    if not marker.exists():
        UNPACK_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(selected_zip, 'r') as zf:
            zf.extractall(UNPACK_DIR)
        marker.write_text(f'extracted from: {selected_zip}
')
    print('Found and extracted zip:', selected_zip)
else:
    print('No dataset zip detected (this is OK if you already have data/ present).')

# Auto-detect where the data lives
# - When running from this folder: data/ sits next to the notebook
# - When running from repo root: challenges/eda-challenge/data
candidates = [
    Path('data'),
    Path('.'),
    Path('challenges/eda-challenge/data'),
    Path('fish_mortality'),  # legacy structure support
    UNPACK_DIR / 'fish_mortality',
    UNPACK_DIR,
]
DATA_DIR = None
for d in candidates:
    if all((d / f).exists() for f in required):
        DATA_DIR = d
        break

if DATA_DIR is None:
    msg = (
        'Could not find the datasets.
'
        '- If you cloned the repo: make sure `data/` exists next to the notebook.
'
        '- If you uploaded a zip: ensure it contains the required files and re-run this cell.
'
        f'Expected files: {required}'
    )
    raise FileNotFoundError(msg)

print('Using DATA_DIR =', DATA_DIR.resolve())

# Optional (Drive): uncomment to mount and set DATA_DIR manually
# if IN_COLAB:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     DATA_DIR = Path('/content/drive/MyDrive/<your-folder>')

## 2. Load the datasets
We load each dataset and parse dates into consistent daily timestamps.

Notes on parsing:
- `river_flow.xml`: we use the built-in etree parser to avoid external `lxml` dependency.
- `nutrients.csv`: dates appear in multiple formats, and units are mixed (`mg/L` and `µg/L`). We normalize units to `mg/L`.

In [ ]:
# Water quality sensors (daily)
water_quality = pd.read_csv(DATA_DIR / 'water_quality_sensors.csv')
water_quality['date'] = pd.to_datetime(water_quality['date'], errors='coerce').dt.floor('D')
water_quality = water_quality.rename(columns={'T (°C)': 'water_temp_c', 'DO (mg/L)': 'do_mg_l'})
water_quality['water_temp_c'] = pd.to_numeric(water_quality['water_temp_c'], errors='coerce')
water_quality['do_mg_l'] = pd.to_numeric(water_quality['do_mg_l'], errors='coerce')

# Meteorology (daily)
meteo = pd.read_json(DATA_DIR / 'meteo.json')
meteo['date'] = pd.to_datetime(meteo['date'], errors='coerce').dt.floor('D')
meteo = meteo.rename(columns={
    'temperature (°C)': 'air_temp_c',
    'rainfall (mm)': 'rain_mm',
    'sunlight (hrs)': 'sun_hrs',
})

# River flow (daily)
river_flow = pd.read_xml(DATA_DIR / 'river_flow.xml', xpath='.//DailyRecord', parser='etree')
river_flow['date'] = pd.to_datetime(river_flow['ObservationDate'], errors='coerce').dt.floor('D')
river_flow['discharge_m3s'] = pd.to_numeric(river_flow['Discharge_m3s'], errors='coerce')
river_flow = river_flow[['date', 'discharge_m3s']]

# Nutrients grab samples (irregular)
nutrients = pd.read_csv(DATA_DIR / 'nutrients.csv')
nutrients['Sample_Date_raw'] = nutrients['Sample_Date'].astype(str)

def parse_mixed_date(s: str):
    s = (s or '').strip()
    if not s:
        return pd.NaT
    for fmt in ('%d/%m/%Y', '%d-%m-%Y', '%Y-%m-%d'):
        try:
            return pd.Timestamp(datetime.strptime(s, fmt).date())
        except ValueError:
            continue
    return pd.NaT

nutrients['date'] = nutrients['Sample_Date_raw'].map(parse_mixed_date)
nutrients['TP_Value'] = pd.to_numeric(nutrients['TP_Value'], errors='coerce')
# Normalize to mg/L (some rows are in µg/L; file may show as '�g/L')
unit = nutrients['Unit'].astype(str)
ug_mask = unit.str.contains('g/L', regex=False) & (~unit.str.contains('mg/L', regex=False))
nutrients['tp_mg_l'] = nutrients['TP_Value']
nutrients.loc[ug_mask, 'tp_mg_l'] = nutrients.loc[ug_mask, 'tp_mg_l'] / 1000.0

display(water_quality.head(3))
display(meteo.head(3))
display(river_flow.head(3))
display(nutrients.head(3))

## 3. Data quality checks & cleaning
We explicitly flag values that are physically impossible or very likely placeholders.

Cleaning rules (conservative):
- Dissolved oxygen (DO) < 0 mg/L is impossible -> set to missing.
- Water temperature > 35°C is extremely unlikely for a river here -> treat as sensor failure -> set to missing.
- Discharge < 0 is invalid -> set to missing.

We keep a short log so conclusions can refer to what was altered.

In [ ]:
clean_log = []

# Water quality cleaning
n_do_neg = int((water_quality['do_mg_l'] < 0).sum())
n_temp_hi = int((water_quality['water_temp_c'] > 35).sum())
if n_do_neg:
    clean_log.append(f'water_quality: set {n_do_neg} rows with DO<0 to NaN')
    water_quality.loc[water_quality['do_mg_l'] < 0, 'do_mg_l'] = np.nan
if n_temp_hi:
    clean_log.append(f'water_quality: set {n_temp_hi} rows with water_temp_c>35 to NaN')
    water_quality.loc[water_quality['water_temp_c'] > 35, 'water_temp_c'] = np.nan

# Flow cleaning
n_dis_neg = int((river_flow['discharge_m3s'] < 0).sum())
if n_dis_neg:
    clean_log.append(f'river_flow: set {n_dis_neg} rows with discharge<0 to NaN')
    river_flow.loc[river_flow['discharge_m3s'] < 0, 'discharge_m3s'] = np.nan

# Nutrients info
n_tp_na = int(nutrients['tp_mg_l'].isna().sum())
n_ug = int(ug_mask.sum())
clean_log.append(f'nutrients: {n_ug} rows converted from µg/L to mg/L; tp_mg_l missing={n_tp_na}')

pd.DataFrame({'cleaning_steps': clean_log})

## 4. Nutrients overview (TP)
Nutrient measurements are grab samples (not daily). We visualize them as points in time, and we look at their distribution after unit normalization.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))

# Time series (scatter)
ax[0].scatter(nutrients['date'], nutrients['tp_mg_l'], s=20, alpha=0.8)
ax[0].set_title('Total Phosphorus (TP) grab samples (mg/L)')
ax[0].set_xlabel('Date')
ax[0].set_ylabel('TP (mg/L)')

# Distribution
sns.histplot(nutrients['tp_mg_l'], bins=30, ax=ax[1])
ax[1].set_title('TP distribution (mg/L)')
ax[1].set_xlabel('TP (mg/L)')

plt.tight_layout()
plt.show()

nutrients['tp_mg_l'].describe()

## 5. Build a daily master table
We join datasets on daily `date`. Since nutrients are sparse, we aggregate nutrients to a daily TP maximum (and mean if multiple samples exist).

We also compute rolling rainfall totals (3d/7d) to quantify runoff pressure.

In [ ]:
nut_daily = (nutrients.dropna(subset=['date'])
             .groupby('date', as_index=False)
             .agg(tp_mean=('tp_mg_l', 'mean'), tp_max=('tp_mg_l', 'max'), tp_n=('tp_mg_l', 'count')))

daily = (meteo[['date', 'air_temp_c', 'rain_mm', 'sun_hrs']]
         .merge(river_flow, on='date', how='outer')
         .merge(water_quality[['date', 'water_temp_c', 'do_mg_l']], on='date', how='outer')
         .merge(nut_daily, on='date', how='outer')
         .sort_values('date')
        )

for k in (3, 7):
    daily[f'rain_{k}d'] = daily['rain_mm'].rolling(k, min_periods=1).sum()

display(daily.head(5))
daily.isna().mean().sort_values(ascending=False).to_frame('missing_fraction').head(12)

### Missingness map
This quick heatmap shows where data is missing over time. It is especially relevant for nutrients (grab samples, so mostly missing) and river discharge (some gaps / invalid placeholders).

In [ ]:
cols = ['discharge_m3s', 'water_temp_c', 'do_mg_l', 'tp_max']
plt.figure(figsize=(14, 2.5))
sns.heatmap(daily[cols].isna().T, cbar=False, xticklabels=False, yticklabels=cols)
plt.title('Missing values over time (white = present, dark = missing)')
plt.xlabel('Date index (2018 -> 2022)')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 6. Global visualizations
First we look at the full timeline to understand seasonality and spot anomalies.

We plot: air temperature, rainfall, discharge, water temperature, dissolved oxygen, and TP (daily max).

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(14, 14), sharex=True)

axes[0].plot(daily['date'], daily['air_temp_c'], lw=1)
axes[0].set_title('Air temperature (°C)')

axes[1].plot(daily['date'], daily['rain_mm'], lw=1)
axes[1].set_title('Rainfall (mm/day)')

axes[2].plot(daily['date'], daily['discharge_m3s'], lw=1)
axes[2].set_title('River discharge (m³/s)')

axes[3].plot(daily['date'], daily['water_temp_c'], lw=1)
axes[3].set_title('Water temperature (°C)')

axes[4].plot(daily['date'], daily['do_mg_l'], lw=1)
axes[4].set_title('Dissolved Oxygen (mg/L)')

axes[5].scatter(daily['date'], daily['tp_max'], s=10, alpha=0.8)
axes[5].set_title('TP daily max (mg/L) (grab samples)')

plt.tight_layout()
plt.show()

We also inspect distributions (after cleaning) to understand typical ranges and rare extremes.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

sns.histplot(daily['water_temp_c'], bins=40, ax=axes[0,0])
axes[0,0].set_title('Water temperature')

sns.histplot(daily['do_mg_l'], bins=40, ax=axes[0,1])
axes[0,1].set_title('Dissolved oxygen')

sns.histplot(daily['discharge_m3s'], bins=40, ax=axes[0,2])
axes[0,2].set_title('Discharge')

sns.histplot(daily['rain_mm'], bins=40, ax=axes[1,0])
axes[1,0].set_title('Rainfall')

sns.histplot(daily['air_temp_c'], bins=40, ax=axes[1,1])
axes[1,1].set_title('Air temperature')

sns.histplot(daily['tp_max'], bins=30, ax=axes[1,2])
axes[1,2].set_title('TP daily max (mg/L)')

plt.tight_layout()
plt.show()

## 2. Timeline of Fish Mortality Events

Based on `fish_mortality_report.txt`, we identified six key incidents:

In [ ]:
# Parse incident report file (reproducible)
txt = (DATA_DIR / 'fish_mortality_report.txt').read_text(encoding='utf-8')
blocks = [b.strip() for b in txt.split('----------------------------------------') if b.strip()]
rows = []
for b in blocks:
    m_date = re.search(r'Date:\s*([0-9]{4}-[0-9]{2}-[0-9]{2})', b)
    m_dead = re.search(r'Estimated dead fish:\s*([0-9]+)', b)
    m_notes = re.search(r'Notes:\s*(.*)', b)
    if m_date:
        rows.append({
            'date': pd.to_datetime(m_date.group(1)),
            'dead_fish_est': int(m_dead.group(1)) if m_dead else None,
            'notes': (m_notes.group(1).strip() if m_notes else None),
        })
incident_df = pd.DataFrame(rows).sort_values('date')
incident_df

In [ ]:
# Visualize fish mortality counts over time
plt.figure(figsize=(10, 3))
plt.plot(incident_df['date'], incident_df['dead_fish_est'], marker='o')
for _, r in incident_df.iterrows():
    plt.annotate(str(r['dead_fish_est']), (r['date'], r['dead_fish_est']), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=9)
plt.title('Reported fish mortality events')
plt.xlabel('Date')
plt.ylabel('Estimated dead fish')
plt.tight_layout()
plt.show()

## 3. Analysis per Incident

For each incident we plot a window around the event date and compute a small evidence table.

Plot design:
- Top: DO + water temperature (two y-axes)
- Middle: discharge
- Bottom: rainfall (bars) + TP (points)

We keep the analysis consistent across events to avoid cherry-picking.

In [ ]:
def incident_window_plot(d: pd.Timestamp, window_days: int = 7):
    start = d - pd.Timedelta(days=window_days)
    end = d + pd.Timedelta(days=window_days)
    w = daily[(daily['date'] >= start) & (daily['date'] <= end)].copy()

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

    # DO + water temp
    ax0 = axes[0]
    ax0.plot(w['date'], w['do_mg_l'], color='tab:blue', label='DO (mg/L)')
    ax0.set_ylabel('DO (mg/L)', color='tab:blue')
    ax0.tick_params(axis='y', labelcolor='tab:blue')
    ax0.axvline(d, color='black', ls='--', lw=1)

    ax0b = ax0.twinx()
    ax0b.plot(w['date'], w['water_temp_c'], color='tab:red', label='Water T (°C)', alpha=0.8)
    ax0b.set_ylabel('Water T (°C)', color='tab:red')
    ax0b.tick_params(axis='y', labelcolor='tab:red')

    ax0.set_title(f'Incident window: {d.date()} (±{window_days}d)')

    # discharge
    ax1 = axes[1]
    ax1.plot(w['date'], w['discharge_m3s'], color='tab:green')
    ax1.axvline(d, color='black', ls='--', lw=1)
    ax1.set_ylabel('Discharge (m³/s)')

    # rainfall + TP
    ax2 = axes[2]
    ax2.bar(w['date'], w['rain_mm'], color='tab:gray', alpha=0.6, label='Rain (mm)')
    ax2.axvline(d, color='black', ls='--', lw=1)
    ax2.set_ylabel('Rain (mm)')

    ax2b = ax2.twinx()
    ax2b.scatter(w['date'], w['tp_max'], color='purple', s=35, alpha=0.9, label='TP max (mg/L)')
    ax2b.set_ylabel('TP max (mg/L)')

    plt.tight_layout()
    plt.show()

    # Evidence summary table
    day = daily[daily['date'] == d]
    day = day.iloc[0] if len(day) == 1 else None

    def w_stat(col, func):
        s = pd.to_numeric(w[col], errors='coerce').dropna()
        return None if s.empty else float(getattr(s, func)())

    summary = {
        'date': str(d.date()),
        'do_min_in_window': w_stat('do_mg_l', 'min'),
        'do_day': None if day is None or pd.isna(day['do_mg_l']) else float(day['do_mg_l']),
        'water_temp_max_in_window': w_stat('water_temp_c', 'max'),
        'water_temp_day': None if day is None or pd.isna(day['water_temp_c']) else float(day['water_temp_c']),
        'discharge_max_in_window': w_stat('discharge_m3s', 'max'),
        'discharge_day': None if day is None or pd.isna(day['discharge_m3s']) else float(day['discharge_m3s']),
        'tp_max_in_window': w_stat('tp_max', 'max'),
        'tp_day': None if day is None or pd.isna(day['tp_max']) else float(day['tp_max']),
        'rain_3d_max_in_window': w_stat('rain_3d', 'max'),
        'rain_7d_max_in_window': w_stat('rain_7d', 'max'),
    }

    return pd.DataFrame([summary])

# Run all incidents and concatenate evidence tables
evidence_tables = []
for d in incident_df['date']:
    evidence_tables.append(incident_window_plot(d, window_days=7))

evidence = pd.concat(evidence_tables, ignore_index=True)
evidence

### Incident 1: July 24, 2019

**Conclusion:** This event was caused by natural factors. A severe heatwave (air temp >32°C) combined with extremely low river flow (4.0 m³/s) led to high water temperatures (24.0°C) and a critical drop in dissolved oxygen (DO) to 3.6 mg/L. This was exacerbated by a massive phosphorus spike (6.19 mg/L), likely triggering an algal bloom that consumed the remaining oxygen.

### Incident 2 & 4: Nov 13, 2019 & Nov 14, 2021

**Conclusion:** These are classic runoff events. Heavy rainfall (e.g., 29.7mm on Nov 12, 2019; 42.5mm on Nov 13, 2021) caused extreme river discharge (>55 m³/s). This corresponds with very high phosphorus levels (5.5-5.9 mg/L). The runoff from surrounding lands, likely contaminated by the Royal Alchemics Combine, washed a high concentration of pollutants into the river, causing the fish mortality.

### Incident 3: August 9, 2021

**Conclusion:** This event is a more extreme version of the runoff scenario. A massive cloudburst the day before (85.0 mm of rain) caused the river flow to double, peaking at 56.0 m³/s. This washed a huge amount of pollutants into the river, evidenced by a phosphorus spike (4.76 mg/L), and triggered a measurable oxygen crash to a lethal 3.1 mg/L.

### Incident 5: June 12, 2022

**Conclusion:** The evidence points to an algal bloom event (eutrophication). A combination of heavy rain (22.2 mm), high sunlight, and a massive phosphorus peak (5.43 mg/L) created ideal conditions. The extremely high DO reading of 15.0 mg/L is a classic sign of oxygen supersaturation caused by photosynthesis from a large algal bloom during the day. This can lead to gas bubble disease in fish and is often followed by a lethal oxygen crash at night when the algae respire.

### Incident 6: July 26, 2022

**Conclusion:** This incident strongly indicates a direct chemical discharge. Unlike the other events, there were no environmental triggers: the weather was stable, river flow was normal, and both DO (7.6 mg/L) and phosphorus (2.2 mg/L) levels were normal. The key evidence is the "unusual foam" mentioned in the report, which is inconsistent with natural events but highly consistent with an industrial or chemical spill. This points directly to an illegal dumping event from the Royal Alchemics Combine.

## 4. Scan for potential unreported events
The mission explicitly asks to remain open to unreported events. We therefore run a *simple* scan for days showing multiple stress signals at once.

Signals used (interpretable and conservative):
- Very low DO (<= 3.5 mg/L)
- High TP (>= 5.0 mg/L)
- Very high discharge (>= 55 m³/s)
- Heavy rainfall (>= 50 mm in a day OR >= 80 mm in 3 days)

We also exclude a ±3 day neighborhood around known incidents to avoid re-discovering the same event.

In [ ]:
inc_dates = set(pd.to_datetime(incident_df['date']).dt.floor('D'))

def near_known_incident(d: pd.Timestamp, k: int = 3) -> bool:
    return any(abs((d - i).days) <= k for i in inc_dates)

cand = daily.copy()
cand = cand[~cand['date'].apply(near_known_incident)]

cand['sig_low_do'] = cand['do_mg_l'].notna() & (cand['do_mg_l'] <= 3.5)
cand['sig_high_tp'] = cand['tp_max'].notna() & (cand['tp_max'] >= 5.0)
cand['sig_high_dis'] = cand['discharge_m3s'].notna() & (cand['discharge_m3s'] >= 55.0)
cand['sig_heavy_rain'] = cand['rain_mm'].notna() & ((cand['rain_mm'] >= 50.0) | (cand['rain_3d'] >= 80.0))

sig_cols = ['sig_low_do', 'sig_high_tp', 'sig_high_dis', 'sig_heavy_rain']
cand['signal_count'] = cand[sig_cols].sum(axis=1)
hits = cand[cand['signal_count'] >= 2].copy()

hits = hits[['date', 'signal_count', 'do_mg_l', 'water_temp_c', 'discharge_m3s', 'tp_max', 'rain_mm', 'rain_3d', 'rain_7d']].sort_values(['signal_count', 'date'], ascending=[False, True])
hits

## 4. Overall Conclusion

The fish mortality in the Green River is not the fault of the Myrtfolk. The evidence points to a systemic pollution problem originating from the **Royal Alchemics Combine**. 

Our analysis reveals two primary mechanisms for the fish kills:
1.  **Pollutant Runoff:** The factory's operations have likely contaminated the surrounding area with phosphorus. During periods of heavy rainfall, this is washed into the river, causing eutrophication, algal blooms, and lethal oxygen crashes.
2.  **Direct Discharge:** At least one incident shows strong evidence of a direct, deliberate chemical discharge into the river, identifiable by the lack of environmental triggers and the presence of unnatural foam.

The Myrtfolk are innocent. The continued operation of the Royal Alchemics Combine without proper environmental controls is the true cause of this ecological disaster.